# Neural Network Explorer — Hands-On Practice Notebook
*From a single neuron to multimodal models — theory recaps + implementations*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nirty/LearningMaterials/blob/main/SystemDesign/ML/nn_explorer_practice.ipynb)

In [ ]:
!pip install -q torch torchvision matplotlib numpy tiktoken

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 1. Building Blocks
*Neurons, activations, layers, loss functions, and backpropagation*

### 1.1 Single Neuron from Scratch

A neuron is the fundamental unit of a neural network. It computes:

$$\text{output} = \text{activation}\left(\sum_{i} w_i \cdot x_i + b\right)$$

where $w_i$ are weights, $x_i$ are inputs, $b$ is the bias, and the activation function introduces non-linearity.

In [ ]:
# Single Neuron from Scratch
def neuron(x, weights, bias, activation='relu'):
    """A single artificial neuron."""
    z = np.dot(x, weights) + bias  # weighted sum + bias

    # Activation functions
    if activation == 'relu':
        return np.maximum(0, z)
    elif activation == 'sigmoid':
        return 1 / (1 + np.exp(-z))
    elif activation == 'tanh':
        return np.tanh(z)
    elif activation == 'linear':
        return z

# Test it
x = np.array([0.8, 0.6])
w = np.array([0.5, -0.3])
b = 0.1

for act in ['linear', 'relu', 'sigmoid', 'tanh']:
    result = neuron(x, w, b, act)
    print(f"  {act:>8}: neuron({x}, {w}, b={b}) = {result:.4f}")

print(f"\nManual check: 0.8*0.5 + 0.6*(-0.3) + 0.1 = {0.8*0.5 + 0.6*(-0.3) + 0.1}")

### 1.2 Activation Functions — Implement & Visualize

Activation functions introduce non-linearity. Without them, stacking layers would just be matrix multiplication (a linear model). Key activations:
- **ReLU**: Simple, fast, but can "die" (output 0 forever)
- **Sigmoid**: Outputs in (0,1), suffers from vanishing gradients
- **Tanh**: Outputs in (-1,1), zero-centered, also saturates
- **GELU**: Smooth approximation used in Transformers

In [ ]:
# Activation Functions from Scratch
def relu(x): return np.maximum(0, x)
def sigmoid(x): return 1 / (1 + np.exp(-x))
def tanh_act(x): return np.tanh(x)
def gelu(x): return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

# Also their derivatives
def relu_grad(x): return (x > 0).astype(float)
def sigmoid_grad(x): s = sigmoid(x); return s * (1 - s)
def tanh_grad(x): return 1 - np.tanh(x)**2
def gelu_grad(x):
    # Approximate derivative
    cdf = 0.5 * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))
    pdf = np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)
    return cdf + x * pdf

x = np.linspace(-5, 5, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Activations
for name, fn, color in [('ReLU', relu, '#34d399'), ('Sigmoid', sigmoid, '#f472b6'),
                          ('Tanh', tanh_act, '#a78bfa'), ('GELU', gelu, '#22d3ee')]:
    axes[0].plot(x, fn(x), label=name, color=color, linewidth=2)
axes[0].set_title('Activation Functions', fontsize=14, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].axhline(y=0, color='gray', linewidth=0.5)
axes[0].axvline(x=0, color='gray', linewidth=0.5); axes[0].set_xlim(-5, 5)

# Derivatives
for name, fn, color in [('ReLU', relu_grad, '#34d399'), ('Sigmoid', sigmoid_grad, '#f472b6'),
                          ('Tanh', tanh_grad, '#a78bfa'), ('GELU', gelu_grad, '#22d3ee')]:
    axes[1].plot(x, fn(x), label=f"{name}'", color=color, linewidth=2)
axes[1].set_title('Derivatives (Gradients)', fontsize=14, fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3); axes[1].axhline(y=0, color='gray', linewidth=0.5)
axes[1].set_xlim(-5, 5)

plt.tight_layout(); plt.show()

print("Key observations:")
print("  - ReLU: gradient is 0 or 1 (dying neuron problem when always 0)")
print("  - Sigmoid/Tanh: gradients vanish near extremes (saturation)")
print("  - GELU: smooth, non-zero gradient everywhere (best for Transformers)")

### 1.3 Build a 2-Layer Network (NumPy)

A 2-layer network chains two linear transformations with an activation in between:

$$\mathbf{a}_1 = \text{ReLU}(\mathbf{x} \cdot W_1 + \mathbf{b}_1)$$
$$\mathbf{z}_2 = \mathbf{a}_1 \cdot W_2 + \mathbf{b}_2$$
$$\hat{\mathbf{y}} = \text{softmax}(\mathbf{z}_2)$$

In [ ]:
# 2-Layer Neural Network — Forward Pass (NumPy)
np.random.seed(42)

# Architecture: 3 inputs → 4 hidden → 2 outputs
W1 = np.random.randn(3, 4) * 0.5  # (3, 4)
b1 = np.zeros(4)                    # (4,)
W2 = np.random.randn(4, 2) * 0.5  # (4, 2)
b2 = np.zeros(2)                    # (2,)

# Forward pass
x = np.array([1.0, 0.5, -0.3])
print(f"Input: {x}, shape: {x.shape}")

# Layer 1
z1 = x @ W1 + b1
a1 = relu(z1)
print(f"\nLayer 1:")
print(f"  z1 = x @ W1 + b1 = {z1}")
print(f"  a1 = relu(z1)    = {a1}")

# Layer 2
z2 = a1 @ W2 + b2
# Softmax for classification
exp_z2 = np.exp(z2 - np.max(z2))  # numerical stability
probs = exp_z2 / exp_z2.sum()
print(f"\nLayer 2:")
print(f"  z2 = a1 @ W2 + b2 = {z2}")
print(f"  softmax(z2) = {probs}")
print(f"  Predicted class: {np.argmax(probs)}")

# Parameter count
total = W1.size + b1.size + W2.size + b2.size
print(f"\nTotal parameters: {total} ({W1.size} + {b1.size} + {W2.size} + {b2.size})")

### 1.4 Loss Functions — MSE vs Cross-Entropy

Loss functions measure how wrong our predictions are:
- **MSE** (Mean Squared Error): $(\hat{y} - y)^2$ — used for regression
- **Cross-Entropy**: $-y \log(\hat{y})$ — used for classification, penalizes confident wrong predictions much more

In [ ]:
# Loss Functions: MSE vs Cross-Entropy
pred = np.linspace(0.01, 0.99, 200)
true_label = 1.0

mse = (pred - true_label) ** 2
ce = -np.log(pred)

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(pred, mse, label='MSE: (\u0177 - y)\u00b2', color='#60a5fa', linewidth=2)
ax.plot(pred, ce, label='Cross-Entropy: -log(\u0177)', color='#f97316', linewidth=2)
ax.set_xlabel('Prediction \u0177 (true label = 1)', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('MSE vs Cross-Entropy Loss (true label = 1)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 5)
plt.tight_layout(); plt.show()

print("When prediction is confidently WRONG (\u0177 \u2192 0):")
print(f"  MSE = {(0.01 - 1)**2:.4f}")
print(f"  CE  = {-np.log(0.01):.4f}  \u2190 much larger gradient = faster learning!")

### 1.5 Backpropagation — Manual vs PyTorch

Backpropagation applies the **chain rule** to compute gradients of the loss w.r.t. each weight:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_1}$$

We compute this manually and verify with PyTorch's autograd.

In [ ]:
# Backpropagation: Manual vs PyTorch Autograd

# Simple network: 1 input → 1 hidden (relu) → 1 output (mse loss)
# Manual computation
x_val, y_val = 2.0, 1.0
w1_val, w2_val = 0.5, -0.3

# Forward
z1 = x_val * w1_val      # 1.0
a1 = max(0, z1)           # 1.0 (relu)
y_hat = a1 * w2_val       # -0.3
loss = (y_hat - y_val)**2 # 1.69

# Backward (chain rule)
dloss_dyhat = 2 * (y_hat - y_val)         # 2 * (-1.3) = -2.6
dyhat_da1 = w2_val                          # -0.3
da1_dz1 = 1.0 if z1 > 0 else 0.0          # 1.0 (relu derivative)
dz1_dw1 = x_val                             # 2.0

dloss_dw2 = dloss_dyhat * a1               # -2.6 * 1.0 = -2.6
dloss_dw1 = dloss_dyhat * dyhat_da1 * da1_dz1 * dz1_dw1  # -2.6 * -0.3 * 1.0 * 2.0 = 1.56

print("Manual Backprop:")
print(f"  Forward: x={x_val} \u2192 z1={z1} \u2192 a1={a1} \u2192 \u0177={y_hat} \u2192 loss={loss}")
print(f"  \u2202loss/\u2202w1 = {dloss_dw1:.4f}")
print(f"  \u2202loss/\u2202w2 = {dloss_dw2:.4f}")

# Verify with PyTorch
w1_t = torch.tensor(w1_val, requires_grad=True)
w2_t = torch.tensor(w2_val, requires_grad=True)
x_t = torch.tensor(x_val)
y_t = torch.tensor(y_val)

z1_t = x_t * w1_t
a1_t = torch.relu(z1_t)
yhat_t = a1_t * w2_t
loss_t = (yhat_t - y_t) ** 2
loss_t.backward()

print(f"\nPyTorch Autograd:")
print(f"  \u2202loss/\u2202w1 = {w1_t.grad.item():.4f}")
print(f"  \u2202loss/\u2202w2 = {w2_t.grad.item():.4f}")
print(f"\n\u2713 Match!" if abs(w1_t.grad.item() - dloss_dw1) < 1e-6 else "\u2717 Mismatch!")

### 1.6 Gradient Descent — Visualize Convergence

Gradient descent updates weights in the direction that decreases loss:

$$w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$$

The **learning rate** $\eta$ is critical: too small = slow, too large = oscillates/diverges.

In [ ]:
# Gradient Descent Visualization
def loss_fn(w): return (w - 0.5) ** 2
def grad_fn(w): return 2 * (w - 0.5)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
learning_rates = [0.01, 0.3, 1.1]
titles = ['Too Slow (lr=0.01)', 'Just Right (lr=0.3)', 'Oscillates! (lr=1.1)']

w_range = np.linspace(-1, 3, 200)

for ax, lr, title in zip(axes, learning_rates, titles):
    ax.plot(w_range, loss_fn(w_range), 'b-', alpha=0.3, linewidth=2)

    # Run gradient descent
    w = 2.0
    trajectory = [w]
    for _ in range(30):
        w = w - lr * grad_fn(w)
        trajectory.append(w)

    # Plot trajectory
    for i in range(len(trajectory)-1):
        ax.plot(trajectory[i], loss_fn(trajectory[i]), 'ro', markersize=4, alpha=0.5+0.5*i/len(trajectory))
        ax.annotate('', xy=(trajectory[i+1], loss_fn(trajectory[i+1])),
                    xytext=(trajectory[i], loss_fn(trajectory[i])),
                    arrowprops=dict(arrowstyle='->', color='red', alpha=0.3))

    ax.plot(0.5, 0, 'g*', markersize=15, label='Target w=0.5')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('w'); ax.set_ylabel('loss')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.1, 3)

plt.tight_layout(); plt.show()

## 2. Convolutional Neural Networks
*Local filters, shared weights, spatial hierarchy*

### 2.1 Convolution from Scratch

A convolution slides a small **kernel** (filter) across the image, computing element-wise products and summing them at each position. This detects local patterns like edges, textures, and shapes.

In [ ]:
# 2D Convolution from Scratch
def conv2d(image, kernel):
    """Apply 2D convolution (no padding, stride=1)."""
    ih, iw = image.shape
    kh, kw = kernel.shape
    oh, ow = ih - kh + 1, iw - kw + 1
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            output[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return output

# Create a simple test image (checkerboard)
image = np.zeros((8, 8))
image[::2, ::2] = 1
image[1::2, 1::2] = 1

# Edge detection kernels
kernels = {
    'Vertical Edge': np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]]),
    'Horizontal Edge': np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]]),
    'Sharpen': np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Input (8\u00d78)')

for ax, (name, kernel) in zip(axes[1:], kernels.items()):
    out = conv2d(image, kernel)
    ax.imshow(out, cmap='RdBu_r'); ax.set_title(f'{name}\n({out.shape[0]}\u00d7{out.shape[1]})')

for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

# Show the math for one position
print("Convolution at position (0,0) with Vertical Edge kernel:")
patch = image[0:3, 0:3]
kernel = kernels['Vertical Edge']
print(f"  Image patch:\n{patch}")
print(f"  Kernel:\n{kernel}")
print(f"  Element-wise multiply:\n{patch * kernel}")
print(f"  Sum = {np.sum(patch * kernel)}")

### 2.2 Max Pooling from Scratch

Max pooling reduces spatial dimensions by taking the maximum value in each local region. This provides:
- **Translation invariance**: small shifts don't change the output
- **Dimensionality reduction**: fewer values to process downstream

In [ ]:
# Max Pooling from Scratch
def max_pool2d(x, pool_size=2, stride=2):
    """Apply max pooling."""
    h, w = x.shape
    oh, ow = h // stride, w // stride
    output = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            patch = x[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(patch)
    return output

# Demo
input_4x4 = np.array([[1, 3, 2, 1],
                       [4, 6, 5, 2],
                       [7, 8, 3, 1],
                       [2, 4, 6, 9]])

pooled = max_pool2d(input_4x4)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(input_4x4, cmap='viridis'); axes[0].set_title('Input 4\u00d74')
for i in range(4):
    for j in range(4):
        axes[0].text(j, i, str(input_4x4[i,j]), ha='center', va='center', color='white', fontweight='bold')

axes[1].imshow(pooled, cmap='viridis'); axes[1].set_title('MaxPool 2\u00d72 \u2192 2\u00d72')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(int(pooled[i,j])), ha='center', va='center', color='white', fontweight='bold')

for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()
print(f"MaxPool result: {pooled.flatten().astype(int).tolist()}")
print(f"Spatial reduction: 4\u00d74 \u2192 2\u00d72 (4x fewer values)")

### 2.3 Build & Train a CNN on MNIST

A complete CNN pipeline: Conv layers extract features, pooling reduces spatial size, and fully-connected layers classify. We train on MNIST (28x28 handwritten digits).

In [ ]:
# CNN on MNIST
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Load MNIST
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_data = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('./data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

# Define CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)   # 28\u00d728\u00d71 \u2192 28\u00d728\u00d716
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)   # 14\u00d714\u00d716 \u2192 14\u00d714\u00d732
        self.pool = nn.MaxPool2d(2, 2)                   # halves spatial dims
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28\u219214
        x = self.pool(F.relu(self.conv2(x)))  # 14\u21927
        x = x.view(-1, 32 * 7 * 7)           # flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {model}")
print(f"Total parameters: {total_params:,}")

# Train for 3 epochs
for epoch in range(3):
    model.train()
    total_loss = 0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Test accuracy
    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            pred = model(data).argmax(dim=1)
            correct += (pred == target).sum().item()

    acc = 100. * correct / len(test_data)
    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Test Accuracy={acc:.1f}%")

### 2.4 Visualize Learned Filters

After training, we can inspect what patterns the CNN learned to detect. Early layers typically learn edges and textures, while deeper layers learn more abstract features.

In [ ]:
# Visualize Conv1 filters
filters = model.conv1.weight.data.cpu()
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    if i < 16:
        ax.imshow(filters[i, 0], cmap='RdBu_r')
    ax.axis('off')
plt.suptitle('Learned Conv1 Filters (16 filters, 3\u00d73)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# Visualize feature maps for a sample image
sample, label = test_data[0]
sample = sample.unsqueeze(0).to(device)

with torch.no_grad():
    feat1 = F.relu(model.conv1(sample))  # after conv1
    feat2 = F.relu(model.conv2(model.pool(feat1)))  # after conv2

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(feat1[0, i].cpu(), cmap='viridis'); axes[0, i].axis('off')
    axes[1, i].imshow(feat2[0, i].cpu(), cmap='viridis'); axes[1, i].axis('off')
axes[0, 0].set_ylabel('Conv1', fontsize=12)
axes[1, 0].set_ylabel('Conv2', fontsize=12)
plt.suptitle(f'Feature Maps for digit "{label}"', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 3. Attention Mechanisms
*The mechanism that lets models focus on what matters*

### 3.1 Scaled Dot-Product Attention

The core attention formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

- **Q** (Query): what am I looking for?
- **K** (Key): what do I contain?
- **V** (Value): what information do I provide?
- Division by $\sqrt{d_k}$ prevents softmax saturation for large dimensions.

In [ ]:
# Scaled Dot-Product Attention from Scratch
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (seq_len, d_k)
    Returns: attention output and weights
    """
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)  # (seq_len, seq_len)

    if mask is not None:
        scores = scores + mask  # -inf for masked positions

    # Softmax
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)

    output = weights @ V  # (seq_len, d_v)
    return output, weights

# Example: 4 tokens, d=8
np.random.seed(42)
seq_len, d_k = 4, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

# Visualize attention weights
fig, ax = plt.subplots(figsize=(5, 4))
tokens = ['Token 0', 'Token 1', 'Token 2', 'Token 3']
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(4)); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(4)); ax.set_yticklabels(tokens)
ax.set_xlabel('Key'); ax.set_ylabel('Query')
ax.set_title('Attention Weights', fontsize=14, fontweight='bold')
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=10)
plt.colorbar(im); plt.tight_layout(); plt.show()

print(f"Each row sums to 1: {weights.sum(axis=1)}")

### 3.2 Multi-Head Attention

Multi-head attention runs multiple attention operations in parallel, each with different learned projections. This allows the model to attend to information from different representation subspaces:

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

In [ ]:
# Multi-Head Attention in PyTorch
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        # Project to Q, K, V
        Q = self.W_Q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)  # (B, h, T, d_k)
        K = self.W_K(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # Attention
        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)

        # Apply to values
        out = (weights @ V).transpose(1, 2).contiguous().view(B, T, C)
        return self.W_O(out), weights

# Test
mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(1, 6, 64)  # batch=1, seq=6, dim=64
out, attn_weights = mha(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention weights: {attn_weights.shape}  (batch, heads, query, key)")

# Visualize per-head attention
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, ax in enumerate(axes):
    ax.imshow(attn_weights[0, i].detach().numpy(), cmap='Blues')
    ax.set_title(f'Head {i+1}', fontweight='bold')
    ax.set_xlabel('Key'); ax.set_ylabel('Query')
plt.suptitle('Multi-Head Attention \u2014 Each Head Learns Different Patterns', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Transformers
*The architecture behind GPT, BERT, ViT, and modern AI*

### 4.1 Positional Encoding

Transformers have no built-in notion of order (unlike RNNs). Positional encodings add position information using sine and cosine functions of different frequencies:

$$PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{model}})$$

In [ ]:
# Sinusoidal Positional Encoding
def sinusoidal_pe(max_len, d_model):
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

pe = sinusoidal_pe(100, 64)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(pe, aspect='auto', cmap='RdBu_r')
axes[0].set_xlabel('Dimension'); axes[0].set_ylabel('Position')
axes[0].set_title('Positional Encoding Matrix', fontweight='bold')

for dim in [0, 1, 10, 20, 40]:
    axes[1].plot(pe[:, dim], label=f'dim {dim}', alpha=0.7)
axes[1].set_xlabel('Position'); axes[1].set_ylabel('Value')
axes[1].set_title('PE Values Across Positions', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print("Low dimensions \u2192 high frequency. High dimensions \u2192 low frequency.")
print("This allows the model to attend to both absolute and relative positions.")

### 4.2 Transformer Encoder Block

A Transformer block consists of:
1. **Multi-Head Self-Attention** (with residual connection + LayerNorm)
2. **Feed-Forward Network** (with residual connection + LayerNorm)

Using Pre-LN (LayerNorm before each sub-layer) for more stable training.

In [ ]:
# Transformer Encoder Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=4, d_ff=512, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LN: LayerNorm \u2192 Attention \u2192 Residual
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=mask)
        x = x + self.dropout(attn_out)

        # Pre-LN: LayerNorm \u2192 FFN \u2192 Residual
        x = x + self.ff(self.ln2(x))
        return x

# Test
block = TransformerBlock(d_model=128, n_heads=4, d_ff=512)
x = torch.randn(2, 10, 128)  # batch=2, seq=10, dim=128
out = block(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}  (shape preserved!)")
print(f"Parameters: {sum(p.numel() for p in block.parameters()):,}")

### 4.3 Causal Mask — What Makes GPT Autoregressive

In decoder-only models (GPT), each token can only attend to previous tokens. This is enforced by a **causal mask** that sets future positions to $-\infty$ before softmax, making their attention weights zero.

In [ ]:
# Causal Mask Visualization
seq_len = 6
tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat']

# Create causal mask (upper triangular = -inf)
causal_mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Show the mask
ax = axes[0]
mask_vis = np.where(causal_mask.numpy() == float('-inf'), 1, 0)
ax.imshow(mask_vis, cmap='Reds', alpha=0.7)
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, '\u2212\u221e' if mask_vis[i,j] else '\u2713', ha='center', va='center', fontsize=9)
ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
ax.set_title('Causal Mask', fontweight='bold')
ax.set_xlabel('Can attend to \u2192')

# Show what each token can see
ax = axes[1]
visibility = 1 - mask_vis
ax.imshow(visibility, cmap='Greens')
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, '\u25cf' if visibility[i,j] else '', ha='center', va='center', fontsize=10)
ax.set_xticks(range(seq_len)); ax.set_xticklabels(tokens, rotation=45)
ax.set_yticks(range(seq_len)); ax.set_yticklabels(tokens)
ax.set_title('What Each Token Sees', fontweight='bold')

plt.tight_layout(); plt.show()
print("'mat' can see all tokens. 'The' can only see itself.")
print("This is what makes GPT autoregressive \u2014 no peeking at the future!")

## 5. Vision Transformer (ViT)
*"An Image is Worth 16\u00d716 Words"*

### 5.1 Patch Embedding — From Image to Sequence

ViT treats an image as a sequence of patches:
1. Split the image into fixed-size patches (e.g., 16\u00d716)
2. Flatten each patch into a vector
3. Project to the model dimension with a linear layer
4. Prepend a [CLS] token and add positional embeddings

In [ ]:
# ViT Patch Embedding
def create_patches(image, patch_size=16):
    """Split image into patches and flatten."""
    C, H, W = image.shape
    n_patches_h = H // patch_size
    n_patches_w = W // patch_size

    # Reshape: (C, H, W) \u2192 (n_patches, patch_size*patch_size*C)
    patches = image.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    patches = patches.contiguous().view(C, -1, patch_size, patch_size)
    patches = patches.permute(1, 0, 2, 3).contiguous().view(-1, C * patch_size * patch_size)
    return patches

# Create a sample "image" (3, 224, 224)
image = torch.randn(3, 224, 224)
patches = create_patches(image, patch_size=16)
print(f"Image shape: {image.shape}")
print(f"Number of patches: {patches.shape[0]}  (14 \u00d7 14 = 196)")
print(f"Each patch: {patches.shape[1]} dims  (16 \u00d7 16 \u00d7 3 = 768)")

# Visualize patching on a real-ish image
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Create a gradient image for visualization
img_vis = np.zeros((224, 224, 3))
img_vis[:,:,0] = np.linspace(0, 1, 224)[np.newaxis, :]
img_vis[:,:,2] = np.linspace(0, 1, 224)[:, np.newaxis]

axes[0].imshow(img_vis); axes[0].set_title('Original Image (224\u00d7224)', fontweight='bold')

# Show grid
axes[1].imshow(img_vis)
for i in range(0, 225, 16):
    axes[1].axhline(y=i, color='white', linewidth=0.5)
    axes[1].axvline(x=i, color='white', linewidth=0.5)
axes[1].set_title('Split into 14\u00d714 = 196 patches', fontweight='bold')

for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

### 5.2 Build a Mini-ViT

A complete Vision Transformer for MNIST classification:
- 28\u00d728 image \u2192 7\u00d77 patches = 16 patches
- Linear projection to 64-dim
- Prepend CLS token \u2192 17 tokens
- 3 Transformer blocks
- CLS token output \u2192 10-class classifier

In [ ]:
# Mini Vision Transformer
class MiniViT(nn.Module):
    def __init__(self, img_size=28, patch_size=7, d_model=64, n_heads=4, n_layers=3, n_classes=10):
        super().__init__()
        self.patch_size = patch_size
        n_patches = (img_size // patch_size) ** 2  # 16 patches for 28/7
        patch_dim = patch_size * patch_size  # 49 for grayscale

        # Patch embedding
        self.patch_embed = nn.Linear(patch_dim, d_model)

        # CLS token + position embeddings
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, d_model))

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_model * 4) for _ in range(n_layers)
        ])

        # Classification head
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size

        # Create patches: (B, n_patches, patch_dim)
        x = x.unfold(2, p, p).unfold(3, p, p)
        x = x.contiguous().view(B, C, -1, p, p).permute(0, 2, 1, 3, 4)
        x = x.contiguous().view(B, -1, C * p * p)

        # Project patches
        x = self.patch_embed(x)  # (B, n_patches, d_model)

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, n_patches+1, d_model)

        # Add position embeddings
        x = x + self.pos_embed

        # Transformer blocks
        for block in self.blocks:
            x = block(x)

        # Extract CLS token \u2192 classify
        cls_out = self.ln(x[:, 0])
        return self.head(cls_out)

vit = MiniViT().to(device)
print(f"Mini-ViT architecture:")
print(f"  Image: 28\u00d728 \u2192 {(28//7)**2} patches of {7*7} \u2192 projected to 64-dim")
print(f"  + CLS token \u2192 {(28//7)**2 + 1} tokens \u00d7 64 dims")
print(f"  3 Transformer blocks (4 heads each)")
print(f"  Total parameters: {sum(p.numel() for p in vit.parameters()):,}")

# Quick test
x = torch.randn(2, 1, 28, 28).to(device)
out = vit(x)
print(f"\n  Input: {x.shape} \u2192 Output: {out.shape}")

### 5.3 Pre-trained ViT — Inference on Real Image

Using a pre-trained ViT-B/16 from torchvision to classify a real image. This model was trained on ImageNet (1000 classes, millions of images).

In [ ]:
# Use Pre-trained ViT from torchvision
from torchvision.models import vit_b_16, ViT_B_16_Weights

weights = ViT_B_16_Weights.DEFAULT
vit_pretrained = vit_b_16(weights=weights).eval()
preprocess = weights.transforms()

# Download a sample image
import urllib.request
from PIL import Image

url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/220px-Cat_November_2010-1a.jpg"
urllib.request.urlretrieve(url, "cat.jpg")
img = Image.open("cat.jpg")

# Preprocess and predict
input_tensor = preprocess(img).unsqueeze(0)
with torch.no_grad():
    output = vit_pretrained(input_tensor)
    probs = F.softmax(output, dim=1)
    top5 = torch.topk(probs, 5)

# Get class names
categories = weights.meta["categories"]
print("ViT-B/16 predictions:")
for prob, idx in zip(top5.values[0], top5.indices[0]):
    print(f"  {categories[idx]:>30}: {prob.item():.1%}")

plt.figure(figsize=(4, 4))
plt.imshow(img); plt.axis('off')
plt.title(f'Predicted: {categories[top5.indices[0][0]]}', fontweight='bold')
plt.show()

## 6. Large Language Models
*Decoder-only transformers predicting the next token*

### 6.1 BPE Tokenization

LLMs don't process raw text — they work with **tokens** (subword units). Byte-Pair Encoding (BPE) iteratively merges frequent character pairs into tokens. Common words become single tokens; rare words are split into pieces.

In [ ]:
# BPE Tokenization with tiktoken
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 encoding

texts = [
    "Hello, world!",
    "The cat sat on the mat.",
    "Supercalifragilisticexpialidocious",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
]

for text in texts:
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f'"{ text}"')
    print(f"  Tokens ({len(tokens)}): {tokens}")
    print(f"  Decoded: {decoded}\n")

print(f"Vocabulary size: {enc.n_vocab:,}")

### 6.2 Build a Mini GPT

A decoder-only Transformer (GPT architecture):
- Token + positional embeddings
- Stack of Transformer blocks with **causal masking**
- Language modeling head predicts the next token
- Autoregressive generation: predict one token at a time

In [ ]:
# Mini GPT (Decoder-Only Transformer)
class MiniGPT(nn.Module):
    def __init__(self, vocab_size=256, d_model=128, n_heads=4, n_layers=4, max_len=128):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_model * 4) for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.max_len = max_len

    def forward(self, idx):
        B, T = idx.shape

        tok_emb = self.token_embed(idx)           # (B, T, d_model)
        pos_emb = self.pos_embed(torch.arange(T, device=idx.device))  # (T, d_model)
        x = tok_emb + pos_emb

        # Causal mask
        mask = torch.triu(torch.ones(T, T, device=idx.device) * float('-inf'), diagonal=1)

        for block in self.blocks:
            x = block(x, mask)

        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_len:]
            logits = self(idx_cond)[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, -1:]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

gpt = MiniGPT().to(device)
total_params = sum(p.numel() for p in gpt.parameters())
print(f"Mini GPT: {total_params:,} parameters")
print(f"  Vocab: 256 (byte-level), d_model=128, 4 heads, 4 layers")

# Test forward pass
tokens = torch.randint(0, 256, (1, 20)).to(device)
logits = gpt(tokens)
print(f"\n  Input tokens: {tokens.shape}")
print(f"  Output logits: {logits.shape}")
print(f"  Next token probabilities shape: {F.softmax(logits[0, -1], dim=-1).shape}")

# Generate some text (untrained - will be random)
seed = torch.tensor([[72, 101, 108, 108, 111]]).to(device)  # "Hello" in ASCII
generated = gpt.generate(seed, max_new_tokens=30, temperature=1.0)
print(f"\nGenerated (untrained): {''.join(chr(t) for t in generated[0].tolist() if 32 <= t < 127)}")

### 6.3 Sampling Strategies — Greedy, Top-k, Top-p

How do LLMs choose the next token from a probability distribution?
- **Greedy**: always pick the most probable token (deterministic)
- **Top-k**: sample from the top k most probable tokens
- **Top-p (Nucleus)**: sample from the smallest set whose cumulative probability exceeds p

In [ ]:
# Sampling Strategies Comparison
logits = torch.tensor([2.0, 1.5, 1.0, 0.5, 0.3, 0.1, -0.5, -1.0, -2.0, -3.0])
tokens = [f"tok_{i}" for i in range(10)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Greedy
probs = F.softmax(logits, dim=0).numpy()
colors = ['#60a5fa' if i == 0 else '#1e293b' for i in range(10)]
axes[0].bar(range(10), probs, color=colors)
axes[0].set_title('Greedy (always pick max)', fontweight='bold')
axes[0].set_xticks(range(10)); axes[0].set_xticklabels(tokens, rotation=45, fontsize=8)

# 2. Top-k (k=3)
k = 3
top_k_vals, top_k_idx = torch.topk(logits, k)
mask = torch.full_like(logits, float('-inf'))
mask[top_k_idx] = logits[top_k_idx]
top_k_probs = F.softmax(mask, dim=0).numpy()
colors = ['#34d399' if top_k_probs[i] > 0 else '#1e293b' for i in range(10)]
axes[1].bar(range(10), top_k_probs, color=colors)
axes[1].set_title('Top-k (k=3)', fontweight='bold')
axes[1].set_xticks(range(10)); axes[1].set_xticklabels(tokens, rotation=45, fontsize=8)

# 3. Top-p / Nucleus (p=0.9)
sorted_probs, sorted_idx = torch.sort(F.softmax(logits, dim=0), descending=True)
cumsum = torch.cumsum(sorted_probs, dim=0)
cutoff = (cumsum <= 0.9).sum().item() + 1
top_p_probs = probs.copy()
threshold = sorted_probs[min(cutoff, len(sorted_probs)-1)].item()
top_p_probs[probs < threshold] = 0
top_p_probs = top_p_probs / top_p_probs.sum()
colors = ['#f472b6' if top_p_probs[i] > 0 else '#1e293b' for i in range(10)]
axes[2].bar(range(10), top_p_probs, color=colors)
axes[2].set_title('Top-p / Nucleus (p=0.9)', fontweight='bold')
axes[2].set_xticks(range(10)); axes[2].set_xticklabels(tokens, rotation=45, fontsize=8)

for ax in axes: ax.set_ylabel('Probability')
plt.tight_layout(); plt.show()

print("Greedy: deterministic, always same output. Good for code/math.")
print("Top-k:  sample from top k tokens. Simple but fixed window.")
print("Top-p:  sample from smallest set whose cumulative prob \u2265 p. Adaptive.")

### 6.4 GPT-2 — Load & Generate

Loading a pre-trained GPT-2 model (124M parameters) and generating text with different sampling strategies.

In [ ]:
!pip install -q transformers

# Load Pre-trained GPT-2
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model_gpt2 = GPT2LMHeadModel.from_pretrained('gpt2').eval()

prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

# Generate with different strategies
strategies = {
    'Greedy (temp=0)': dict(do_sample=False, max_new_tokens=40),
    'Top-k (k=50, temp=0.7)': dict(do_sample=True, top_k=50, temperature=0.7, max_new_tokens=40),
    'Top-p (p=0.9, temp=1.0)': dict(do_sample=True, top_p=0.9, temperature=1.0, max_new_tokens=40),
}

print(f"Prompt: \"{prompt}\"\n")
for name, kwargs in strategies.items():
    with torch.no_grad():
        output = model_gpt2.generate(input_ids, **kwargs, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"[{name}]")
    print(f"  {text}\n")

## 7. Vision-Language & Multimodal
*Models that see AND speak*

### 7.1 CLIP — Zero-Shot Classification

CLIP (Contrastive Language-Image Pre-training) learns a shared embedding space for images and text. It can classify images using natural language descriptions — without any task-specific training (zero-shot).

In [ ]:
# CLIP: Zero-Shot Image Classification
!pip install -q open_clip_torch

import open_clip
from PIL import Image
import urllib.request

# Load CLIP model
clip_model, _, preprocess_clip = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
tokenizer_clip = open_clip.get_tokenizer('ViT-B-32')
clip_model.eval()

# Download a test image
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/220px-Cat_November_2010-1a.jpg"
urllib.request.urlretrieve(url, "test_image.jpg")
image = preprocess_clip(Image.open("test_image.jpg")).unsqueeze(0)

# Define text labels (zero-shot \u2014 no training needed!)
labels = ["a photo of a cat", "a photo of a dog", "a photo of a bird",
          "a photo of a car", "a photo of a flower", "a landscape photo"]
text = tokenizer_clip(labels)

# Compute similarity
with torch.no_grad():
    image_features = clip_model.encode_image(image)
    text_features = clip_model.encode_text(text)

    # Normalize
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    # Cosine similarity
    similarity = (image_features @ text_features.T).squeeze(0)
    probs = F.softmax(similarity * 100, dim=0)  # temperature=100 (CLIP default)

# Display results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(Image.open("test_image.jpg")); axes[0].axis('off')
axes[0].set_title('Input Image', fontweight='bold')

colors = ['#60a5fa' if i == probs.argmax() else '#64748b' for i in range(len(labels))]
axes[1].barh(range(len(labels)), probs.numpy(), color=colors)
axes[1].set_yticks(range(len(labels))); axes[1].set_yticklabels(labels)
axes[1].set_xlabel('Probability'); axes[1].set_title('Zero-Shot Classification (no training!)', fontweight='bold')
plt.tight_layout(); plt.show()

print("CLIP classifies images using natural language descriptions \u2014 no training needed!")

### 7.2 Image-Text Similarity Matrix

CLIP embeds images and text into the same vector space. We can compute cosine similarity between any text descriptions to see how the embedding space captures semantic relationships.

In [ ]:
# CLIP: Image-Text Similarity Matrix
# We'll use multiple text descriptions to explore the embedding space
descriptions = [
    "a cat sitting on a mat",
    "a sunny day at the beach",
    "a red sports car on a highway",
    "a plate of fresh sushi"
]

text_tokens = tokenizer_clip(descriptions)
with torch.no_grad():
    text_feats = clip_model.encode_text(text_tokens)
    text_feats /= text_feats.norm(dim=-1, keepdim=True)

# Compute text-text similarity (demonstrates embedding space)
sim_matrix = (text_feats @ text_feats.T).numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='Blues', vmin=0, vmax=1)
short_labels = ['cat on mat', 'beach day', 'sports car', 'sushi plate']
ax.set_xticks(range(4)); ax.set_xticklabels(short_labels, rotation=45, ha='right')
ax.set_yticks(range(4)); ax.set_yticklabels(short_labels)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=12)
plt.colorbar(im); ax.set_title('Text Embedding Similarity (CLIP)', fontweight='bold')
plt.tight_layout(); plt.show()

print("Diagonal is high (same text = similar). Off-diagonal shows semantic relationships.")

### 7.3 Build a Mini VLM (Vision-Language Model)

A simplified Vision-Language Model architecture:
1. **Image encoder**: extracts visual features (normally a frozen ViT)
2. **Projection layer**: bridges vision and language dimensions, creating "visual tokens"
3. **Text decoder**: processes interleaved visual + text tokens to generate output

This is the core idea behind models like LLaVA, Flamingo, and GPT-4V.

In [ ]:
# Mini Vision-Language Model
class MiniVLM(nn.Module):
    """Simplified VLM: Frozen image encoder + projection + text decoder."""
    def __init__(self, image_dim=512, text_dim=128, vocab_size=256, n_visual_tokens=4):
        super().__init__()
        # Image "encoder" (simplified \u2014 normally a frozen ViT)
        self.image_encoder = nn.Sequential(
            nn.Linear(image_dim, 256), nn.GELU(), nn.Linear(256, image_dim)
        )

        # Projection: bridge vision \u2192 language dimensions
        self.projection = nn.Linear(image_dim, text_dim * n_visual_tokens)
        self.n_visual_tokens = n_visual_tokens

        # Text decoder (simplified)
        self.token_embed = nn.Embedding(vocab_size, text_dim)
        self.decoder_block = TransformerBlock(text_dim, n_heads=4, d_ff=text_dim*4)
        self.lm_head = nn.Linear(text_dim, vocab_size)
        self.text_dim = text_dim

    def forward(self, image_features, text_tokens):
        B = image_features.shape[0]

        # Encode image \u2192 visual tokens
        img_enc = self.image_encoder(image_features)  # (B, image_dim)
        visual_tokens = self.projection(img_enc)       # (B, text_dim * n_visual)
        visual_tokens = visual_tokens.view(B, self.n_visual_tokens, self.text_dim)

        # Embed text tokens
        text_emb = self.token_embed(text_tokens)  # (B, T, text_dim)

        # Interleave: [visual_tokens, text_tokens]
        combined = torch.cat([visual_tokens, text_emb], dim=1)  # (B, n_visual+T, text_dim)

        # Decode
        decoded = self.decoder_block(combined)
        logits = self.lm_head(decoded)
        return logits

vlm = MiniVLM()
print(f"Mini VLM architecture:")
print(f"  Image encoder: Linear(512\u2192256\u2192512)")
print(f"  Projection: Linear(512\u2192512) \u2192 reshape to 4 visual tokens of 128-dim")
print(f"  Decoder: TransformerBlock(128-dim, 4 heads)")
print(f"  Total params: {sum(p.numel() for p in vlm.parameters()):,}")

# Test
fake_image = torch.randn(2, 512)
fake_text = torch.randint(0, 256, (2, 10))
output = vlm(fake_image, fake_text)
print(f"\n  Image features: {fake_image.shape}")
print(f"  Text tokens: {fake_text.shape}")
print(f"  Output logits: {output.shape}  (4 visual + 10 text = 14 positions)")

## Bonus: CNN vs ViT Comparison on MNIST

A head-to-head comparison: does the CNN's inductive bias (locality, translation invariance) help on a small dataset like MNIST, compared to ViT which needs to learn spatial structure from scratch?

In [ ]:
# Train Mini-ViT on MNIST and compare with CNN
print("Training Mini-ViT on MNIST...")
vit_model = MiniViT(img_size=28, patch_size=7, d_model=64, n_heads=4, n_layers=3, n_classes=10).to(device)
vit_optimizer = torch.optim.Adam(vit_model.parameters(), lr=0.001)

vit_accs = []
cnn_model = SimpleCNN().to(device)
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
cnn_accs = []

for epoch in range(5):
    for model_name, m, opt in [('ViT', vit_model, vit_optimizer), ('CNN', cnn_model, cnn_optimizer)]:
        m.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            opt.zero_grad()
            output = m(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            opt.step()
            if batch_idx >= 200: break  # Limit batches for speed

        m.eval()
        correct = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                correct += (m(data).argmax(1) == target).sum().item()
        acc = 100. * correct / len(test_data)

        if model_name == 'ViT': vit_accs.append(acc)
        else: cnn_accs.append(acc)

    print(f"Epoch {epoch+1}: CNN={cnn_accs[-1]:.1f}%, ViT={vit_accs[-1]:.1f}%")

# Plot comparison
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 6), cnn_accs, 'o-', label=f'CNN ({sum(p.numel() for p in cnn_model.parameters()):,} params)', color='#60a5fa')
ax.plot(range(1, 6), vit_accs, 's-', label=f'ViT ({sum(p.numel() for p in vit_model.parameters()):,} params)', color='#f472b6')
ax.set_xlabel('Epoch'); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('CNN vs ViT on MNIST', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("\nKey insight: On small datasets like MNIST, CNN's inductive bias")
print("(locality, translation invariance) gives it an advantage over ViT.")
print("ViT needs much more data to match CNN performance.")

---
## Summary

| Section | Key Concept | Implementation |
|---------|------------|----------------|
| 1. Building Blocks | Neurons, activations, backprop | NumPy from scratch \u2192 PyTorch autograd |
| 2. CNNs | Convolution, pooling, training | Manual conv \u2192 PyTorch CNN on MNIST |
| 3. Attention | Scaled dot-product, multi-head | NumPy attention \u2192 PyTorch MHA |
| 4. Transformers | Encoder block, causal mask | Full TransformerBlock implementation |
| 5. ViT | Patches, CLS token, position | Mini-ViT + pre-trained ViT inference |
| 6. LLMs | Tokenization, generation | Mini-GPT + GPT-2 generation |
| 7. Multimodal | CLIP, VLM architecture | Zero-shot CLIP + Mini VLM |

*From a single neuron to multimodal models \u2014 every layer builds on the last.*